In [ ]:
"""
TorchEEG HMC sleep-staging example
- Loads .edf + .sleepscoring.edf pairs from ./HMC/recordings
- Applies channel picking, band-pass, resampling, normalization
- Builds train/val split by subject ID (SNxxx)
- Trains a lightweight 1D CNN classifier on 30s windows

Requirements (tested versions are examples):
    torch>=2.2
    torcheeg>=1.1.2
    mne>=1.6.1 (pulled by torcheeg for EDF)

Directory structure:
HMC/
└── recordings/
    ├── SN001.edf
    ├── SN001.sleepscoring.edf
    ├── SN002.edf
    ├── SN002.sleepscoring.edf
    └── ...
"""
from __future__ import annotations

import os
import re
import math
import random
from collections import Counter
from typing import List, Dict, Tuple

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

from torcheeg.datasets import HMCDataset
from torcheeg import transforms

# -------------------------
# Reproducibility helpers
# -------------------------

def seed_everything(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(2024)

# -------------------------
# Config
# -------------------------
ROOT = r'C:\Users\RAZER\Documents\Datasets\Sleep_Dataset\physionet.org\files\hmc-sleep-staging\1.1\recordings'
SAMPLE_RATE = 100  # Hz after resampling
CHANNELS = ['EEG F4-M1', 'EEG C4-M1', 'EEG O2-M1', 'EEG C3-M2']
BATCH_SIZE = 128
NUM_WORKERS = max(1, os.cpu_count() // 2)
IO_PATH = './_cache/hmc_lmdb'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Sleep stage map (W=0, N1=1, N2=2, N3=3, REM=4)
LABEL_MAP = {
    'Sleep stage W': 0,
    'Sleep stage N1': 1,
    'Sleep stage N2': 2,
    'Sleep stage N3': 3,
    'Sleep stage R': 4,
    # Some HMC annotations include a lights-off marker tied to a channel; map to Wake.
    'Lights off@@EEG F4-A1': 0,
}

# -------------------------
# Dataset + transforms
# -------------------------

online_t = transforms.Compose([
    transforms.MeanStdNormalize(),  # channel-wise
    transforms.ToTensor(),          # -> torch.float32 [C,T]
])

# Optional: if you want offline caching transforms (e.g., z-score before LMDB write)
offline_t = None

print('Indexing dataset (this may build a cache on first run)...')
dataset = HMCDataset(
    root_path=ROOT,
    sfreq=SAMPLE_RATE,
    channels=CHANNELS,
    l_freq=0.5,
    h_freq=30.0,
    online_transform=online_t,
    offline_transform=offline_t,
    label_transform=transforms.Compose([
        transforms.Select('label'),
        transforms.Mapping(LABEL_MAP)
    ]),
    io_path=IO_PATH,
    io_size=1_024 * 1_024 * 1024,  # 1 GiB LMDB map size (tune to your disk)
    io_mode='lmdb',
    num_worker=0,   # HMCDataset builds the cache in-process; DataLoader workers set below
    verbose=True,
)

# Peek one sample
eeg0, y0 = dataset[0]
print('Sample shape:', tuple(eeg0.shape), 'label:', int(y0))  # (4, 3000), 30s * 100Hz

# -------------------------
# Subject-wise split helper
# -------------------------

def extract_subject_id(info: Dict) -> str:
    """Pulls SNxxx from file stem within dataset's internal info dict."""
    # Info dict is available via dataset[i][2] if using return info; HMCDataset returns (x,y) by default.
    # Workaround: parse from index mapping in dataset.io_path by introspecting private fields (stable enough).
    # Safer approach: regroup indices by file path embedded in dataset.records.
    # torcheeg datasets expose dataset.records: List[Dict]
    return None

# Build indices grouped by source recording path
records = dataset.records  # list of dicts with keys like 'file', 'start', 'end', etc.
by_subject: Dict[str, List[int]] = {}
for idx, rec in enumerate(records):
    # rec['file'] points to the EDF file path; extract SN number
    stem = os.path.basename(rec.get('file', ''))
    m = re.match(r'(SN\d+)', stem)
    sid = m.group(1) if m else stem
    by_subject.setdefault(sid, []).append(idx)

subjects = sorted(by_subject.keys())
print(f'Found {len(subjects)} subjects: {subjects[:10]} ...')

# Simple 80/20 subject split
n_train = max(1, int(0.8 * len(subjects)))
train_subjects = set(subjects[:n_train])
val_subjects = set(subjects[n_train:])

train_indices = [i for s in train_subjects for i in by_subject[s]]
val_indices = [i for s in val_subjects for i in by_subject[s]]

train_set = Subset(dataset, train_indices)
val_set = Subset(dataset, val_indices)

print(f'Train windows: {len(train_set)} | Val windows: {len(val_set)}')

# -------------------------
# Dataloaders
# -------------------------

def collate(batch):
    xs, ys = [], []
    for x, y in batch:
        xs.append(x)
        ys.append(int(y))
    x = torch.stack(xs)  # [B,C,T]
    y = torch.tensor(ys, dtype=torch.long)
    return x, y

train_loader = DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate
)
val_loader = DataLoader(
    val_set, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate
)

# -------------------------
# Model: lightweight 1D CNN (sleep staging baseline)
# -------------------------

class TinySleepNet(nn.Module):
    def __init__(self, in_ch: int = 4, n_classes: int = 5):
        super().__init__()
        self.fe = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, n_classes)
        )
    def forward(self, x):  # x: [B,C,T]
        z = self.fe(x)
        return self.classifier(z)

model = TinySleepNet(in_ch=len(CHANNELS), n_classes=5).to(DEVICE)

# Class weights to mitigate imbalance
stage_counts = Counter()
for _, y in DataLoader(train_set, batch_size=512, collate_fn=collate):
    stage_counts.update(y.tolist())
print('Stage counts:', stage_counts)

max_c = max(stage_counts.values())
weights = torch.tensor([max_c / (stage_counts[i] if stage_counts[i] > 0 else 1) for i in range(5)], dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

# -------------------------
# Train / eval loops
# -------------------------

def run_epoch(loader: DataLoader, train: bool = True) -> Tuple[float, float]:
    model.train(train)
    total_loss, total_correct, total_n = 0.0, 0, 0
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            logits = model(x)
            loss = criterion(logits, y)
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        total_loss += float(loss) * x.size(0)
        total_correct += (logits.argmax(1) == y).sum().item()
        total_n += x.size(0)
    if train:
        scheduler.step()
    return total_loss / max(1, total_n), total_correct / max(1, total_n)

EPOCHS = 10
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    print(f'Epoch {epoch:02d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | val loss {va_loss:.4f} acc {va_acc:.3f}')

print('Done.')

# -------------------------
# Notes / Tips
# -------------------------
# * The HMCDataset already performs EDF reading, band-pass (l_freq/h_freq), and resampling to `sfreq`.
# * Each sample is a 30s epoch: shape [len(CHANNELS), 30*sfreq]. Adjust augmentations in online_transform if needed.
# * To speed up first-run indexing, ensure IO_PATH is on a fast SSD and io_size is ample (e.g., 2–8 GiB for large corpora).
# * For subject-wise CV, replace the simple 80/20 split with K-fold splitting on `by_subject` keys.
# * To add EOG/EMG channels, include them in CHANNELS and bump `in_ch` in TinySleepNet.
